In [ ]:
# SCRIPT 2: ENTRENAMIENTO CON CONTROL ESTRICTO DE THRESHOLD
# Objetivo: Lograr R~74%, P~5% similar al baseline, controlando agresivamente el threshold

import warnings, os, gc, time, joblib, pickle
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
    precision_recall_curve, average_precision_score, confusion_matrix,
    classification_report, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV

try:
    from lightgbm import LGBMClassifier
    import lightgbm as lgb
    print("✅ LightGBM disponible")
except ImportError:
    print("❌ LightGBM no disponible")
    raise SystemExit(1)



✅ LightGBM disponible
🚀 SCRIPT 2: ENTRENAMIENTO CON THRESHOLD CONTROLADO


In [4]:
print("🚀 SCRIPT 2: ENTRENAMIENTO CON THRESHOLD CONTROLADO")
print("=" * 70)

# ==========================
# CONFIG MEJORADA
# ==========================
INPUT_DIR   = "../data/processed_ml"
OUTPUT_DIR  = "../models"
RESULTS_DIR = "../results"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_STATE = 42
TARGET_RECALL = 0.74    # Objetivo similar al baseline
TARGET_PRECISION = 0.05 # Objetivo similar al baseline
MAX_POS_RATE = 0.10     # Máximo 10% de predicciones positivas

def log_progress(msg): 
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

start_time = time.time()

# ==========================
# 1) CARGAR DATOS
# ==========================
log_progress("Cargando datos procesados...")

input_path = os.path.join(INPUT_DIR, "training_data_ready.joblib")
if not os.path.exists(input_path):
    print(f"❌ ERROR: No se encuentra {input_path}")
    raise SystemExit(1)

bundle = joblib.load(input_path)
X_train, y_train = bundle['X_train'], bundle['y_train']
X_val, y_val = bundle['X_val'], bundle['y_val']
X_test, y_test = bundle['X_test'], bundle['y_test']

scaler = bundle.get('scaler')
selector = bundle.get('selector')
selected_features = bundle.get('selected_features')
feature_cols = bundle.get('feature_cols')
cluster_geometries = bundle.get('cluster_geometries')
historical_data = bundle.get('historical_data')
metadata = bundle.get('metadata', {})

log_progress(f"Train: {len(y_train):,} | Val: {len(y_val):,} | Test: {len(y_test):,}")
log_progress(f"Tasa accidentes - Train: {y_train.mean()*100:.2f}% | Val: {y_val.mean()*100:.2f}%")

# ==========================
# 2) HIPERPARÁMETROS MÁS CONSERVADORES
# ==========================
log_progress("Búsqueda de hiperparámetros más conservadores...")

# Parámetros que favorecen menos overfitting
param_distributions = {
    "n_estimators":      [100, 150, 200],  # Menos árboles
    "learning_rate":     [0.01, 0.03, 0.05],  # Learning rate más bajo
    "num_leaves":        [15, 31, 50],     # Menos hojas
    "min_child_samples": [50, 100, 200],   # Más muestras por hoja
    "reg_alpha":         [1.0, 5.0, 10.0], # Más regularización L1
    "reg_lambda":        [1.0, 5.0, 10.0], # Más regularización L2
    "subsample":         [0.7, 0.8],       # Menos subsampling
    "colsample_bytree":  [0.7, 0.8],       # Menos features por árbol
    "min_split_gain":    [0.1, 0.5, 1.0],  # Mayor ganancia para split
}

base_model = LGBMClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
    importance_type='gain',
    objective='binary'  # Explícitamente binario
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rs = RandomizedSearchCV(
    base_model,
    param_distributions=param_distributions,
    n_iter=12,  # Menos iteraciones para terminar más rápido
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE
)

rs.fit(X_train, y_train)
log_progress(f"✅ Mejor score CV (AP): {rs.best_score_:.4f}")

# ==========================
# 3) REENTRENAMIENTO CON EARLY STOPPING AGRESIVO
# ==========================
log_progress("Reentrenamiento con early stopping agresivo...")

best_model = LGBMClassifier(
    **rs.best_params_,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

# Early stopping más agresivo
best_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="binary_logloss",
    callbacks=[
        lgb.early_stopping(stopping_rounds=15),  # Más agresivo
        lgb.log_evaluation(period=0)
    ]
)

actual_iterations = getattr(best_model, 'best_iteration_', len(best_model.evals_result_['valid_0']['binary_logloss']))
log_progress(f"Entrenamiento completado en {actual_iterations} iteraciones")

# ==========================
# 4) CALIBRACIÓN CONSERVADORA
# ==========================
log_progress("Calibrando probabilidades...")

# Usar isotonic en lugar de sigmoid para mejor calibración
calibrated_model = CalibratedClassifierCV(
    estimator=best_model,
    method='isotonic',  # Mejor para probabilidades extremas
    cv=3
)
calibrated_model.fit(X_train, y_train)

# ==========================
# 5) OPTIMIZACIÓN DE THRESHOLD MÚLTIPLE ESTRATEGIAS
# ==========================
log_progress("Optimizando threshold con múltiples estrategias...")

proba_val = calibrated_model.predict_proba(X_val)[:, 1]

# Estadísticas de probabilidades para diagnóstico
log_progress(f"Probabilidades val - Min: {proba_val.min():.6f}, Max: {proba_val.max():.6f}")
log_progress(f"Percentiles - 90%: {np.percentile(proba_val, 90):.6f}, 95%: {np.percentile(proba_val, 95):.6f}, 99%: {np.percentile(proba_val, 99):.6f}")

# Estrategia 1: Threshold basado en percentiles para controlar tasa de positivos
thresholds_percentile = [
    np.percentile(proba_val, 90),  # Top 10%
    np.percentile(proba_val, 95),  # Top 5%
    np.percentile(proba_val, 97),  # Top 3%
    np.percentile(proba_val, 99),  # Top 1%
]

# Estrategia 2: Curva precision-recall tradicional
precisions, recalls, thresholds_pr = precision_recall_curve(y_val, proba_val)

# Evaluar todas las opciones
candidates = []

# Opción 1: Percentiles
for i, thr in enumerate(thresholds_percentile):
    y_pred_temp = (proba_val >= thr).astype(int)
    if y_pred_temp.sum() > 0:  # Evitar división por cero
        p = precision_score(y_val, y_pred_temp)
        r = recall_score(y_val, y_pred_temp)
        pos_rate = y_pred_temp.mean()
        candidates.append({
            'threshold': thr,
            'precision': p,
            'recall': r,
            'pos_rate': pos_rate,
            'f1': 2*p*r/(p+r) if (p+r) > 0 else 0,
            'method': f'percentile_{90+i*2}'
        })

# Opción 2: Precision-Recall curve (filtrar razonables)
for i, thr in enumerate(thresholds_pr[::len(thresholds_pr)//20]):  # Cada 20 puntos
    if i >= len(precisions) - 1:
        break
    p = precisions[i*len(thresholds_pr)//20]
    r = recalls[i*len(thresholds_pr)//20]
    pos_rate = (proba_val >= thr).mean()
    
    if 0.01 <= pos_rate <= MAX_POS_RATE:  # Solo candidatos razonables
        candidates.append({
            'threshold': thr,
            'precision': p,
            'recall': r,
            'pos_rate': pos_rate,
            'f1': 2*p*r/(p+r) if (p+r) > 0 else 0,
            'method': 'pr_curve'
        })

# Seleccionar el mejor candidato
log_progress(f"Evaluando {len(candidates)} candidatos de threshold...")

if len(candidates) == 0:
    # Fallback extremo: usar top 5%
    optimal_threshold = np.percentile(proba_val, 95)
    log_progress("WARNING: Usando fallback percentil 95%")
else:
    # Filtrar candidatos que están cerca del objetivo
    good_candidates = []
    for c in candidates:
        # Penalizar desviaciones del objetivo
        recall_penalty = abs(c['recall'] - TARGET_RECALL) / TARGET_RECALL
        precision_bonus = c['precision'] / TARGET_PRECISION if TARGET_PRECISION > 0 else 1
        
        # Score combinado (menor es mejor para penalty, mayor es mejor para bonus)
        score = c['f1'] * precision_bonus / (1 + recall_penalty)
        c['score'] = score
        
        # Filtros mínimos
        if c['recall'] >= 0.5 and c['precision'] >= 0.01 and c['pos_rate'] <= MAX_POS_RATE:
            good_candidates.append(c)
    
    if len(good_candidates) > 0:
        best_candidate = max(good_candidates, key=lambda x: x['score'])
        optimal_threshold = best_candidate['threshold']
        log_progress(f"Mejor candidato: {best_candidate['method']}")
    else:
        # Último recurso: balance F1
        best_candidate = max(candidates, key=lambda x: x['f1'])
        optimal_threshold = best_candidate['threshold']
        log_progress(f"Usando mejor F1: {best_candidate['method']}")

# Métricas finales en validación
y_pred_val = (proba_val >= optimal_threshold).astype(int)
val_precision = precision_score(y_val, y_pred_val)
val_recall = recall_score(y_val, y_pred_val)
val_pos_rate = y_pred_val.mean()

log_progress(f"Threshold final: {optimal_threshold:.6f}")
log_progress(f"Val → P={val_precision:.3f} | R={val_recall:.3f} | pos_rate={val_pos_rate*100:.1f}%")

# ==========================
# 6) EVALUACIÓN EN TEST
# ==========================
log_progress("Evaluación final en test...")

proba_test = calibrated_model.predict_proba(X_test)[:, 1]
y_pred_test = (proba_test >= optimal_threshold).astype(int)

test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_f05 = fbeta_score(y_test, y_pred_test, beta=0.5)
test_pr_auc = average_precision_score(y_test, proba_test)
test_roc_auc = roc_auc_score(y_test, proba_test)

cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = map(int, cm.ravel())

# Classification report completo
test_report = classification_report(y_test, y_pred_test, 
                                  target_names=['No Accidente', 'Accidente'], 
                                  output_dict=True)

print("\n" + "="*70)
print("📊 CLASSIFICATION REPORT COMPLETO")
print("="*70)
print(classification_report(y_test, y_pred_test, 
                           target_names=['No Accidente', 'Accidente']))

print(f"\n🎯 MÉTRICAS DETALLADAS:")
print(f"   Test Precision:  {test_precision:.3f} ({test_precision*100:.1f}%)")
print(f"   Test Recall:     {test_recall:.3f} ({test_recall*100:.1f}%)")
print(f"   Test F1-Score:   {test_f1:.3f}")
print(f"   Test F0.5-Score: {test_f05:.3f}")
print(f"   PR-AUC:          {test_pr_auc:.4f}")
print(f"   ROC-AUC:         {test_roc_auc:.4f}")

print(f"\n🔢 MATRIZ DE CONFUSIÓN:")
print(f"   TN: {tn:,} | FP: {fp:,}")
print(f"   FN: {fn:,} | TP: {tp:,}")
print(f"   Tasa pred. positivas: {(fp+tp)/(tn+fp+fn+tp)*100:.1f}%")

print(f"\n⚖️ COMPARACIÓN CON OBJETIVOS:")
print(f"   Objetivo Recall ~74%: {'✅' if 0.65 <= test_recall <= 0.85 else '⚠️'} (Actual: {test_recall*100:.1f}%)")
print(f"   Objetivo Precision ~5%: {'✅' if test_precision >= 0.03 else '⚠️'} (Actual: {test_precision*100:.1f}%)")

# ==========================
# 7) GUARDAR MODELO
# ==========================
log_progress("Guardando modelo optimizado...")

streamlit_model = {
    "model": calibrated_model,
    "scaler": scaler,
    "selector": selector,
    "selected_features": selected_features,
    "feature_cols": feature_cols,
    "optimal_threshold": float(optimal_threshold),
    "cluster_geometries": cluster_geometries,
    "historical_data": historical_data,
    "performance": {
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1_score": float(test_f1),
        "f05_score": float(test_f05),
        "pr_auc": float(test_pr_auc),
        "roc_auc": float(test_roc_auc),
        "confusion_matrix": {"tn": tn, "fp": fp, "fn": fn, "tp": tp}
    },
    "config": {
        "target_recall": TARGET_RECALL,
        "target_precision": TARGET_PRECISION,
        "max_pos_rate": MAX_POS_RATE,
        "created_at": datetime.now().isoformat(),
        "model_version": "2.2-threshold-controlled"
    }
}

# Guardar
model_path = os.path.join(OUTPUT_DIR, "barcelona_accident_model_v2.joblib")
joblib.dump(streamlit_model, model_path, compress=3)

total_time = (time.time() - start_time) / 60

print(f"\n⏱️ Tiempo total: {total_time:.1f} min")
print(f"💾 Modelo guardado: {model_path}")
print(f"\n🚀 LISTO PARA STREAMLIT!")

# Limpieza
del X_train, y_train, X_val, y_val, X_test, y_test
gc.collect()

log_progress("✅ Script completado exitosamente")

🚀 SCRIPT 2: ENTRENAMIENTO CON THRESHOLD CONTROLADO
[11:24:34] Cargando datos procesados...
[11:24:34] Train: 704,254 | Val: 442,951 | Test: 447,612
[11:24:34] Tasa accidentes - Train: 7.41% | Val: 1.74%
[11:24:34] Búsqueda de hiperparámetros más conservadores...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
[11:34:49] ✅ Mejor score CV (AP): 0.2773
[11:34:49] Reentrenamiento con early stopping agresivo...
Training until validation scores don't improve for 15 rounds
Did not meet early stopping. Best iteration is:
[200]	valid_0's binary_logloss: 0.520224
[11:35:05] Entrenamiento completado en 200 iteraciones
[11:35:05] Calibrando probabilidades...
[11:35:34] Optimizando threshold con múltiples estrategias...
[11:35:42] Probabilidades val - Min: 0.037764, Max: 1.000000
[11:35:42] Percentiles - 90%: 0.081834, 95%: 0.081834, 99%: 0.081834
[11:35:43] Evaluando 4 candidatos de threshold...
[11:35:43] Usando mejor F1: percentile_90
[11:35:43] Threshold final: 0.081834
[11:35:43] 